# OpenPlaque RCA Centerline Validation — Run All
Choose **Runtime → Run all**. This version mounts Google Drive **first**, uses the existing OpenPlaque data under `MyDrive/OpenPlaque`, loads the RCA CCTA directly from `Full_DICOM.zip`, finds the matching saved RCA refined mask, extracts the centerline, and displays the 0/10/50 mm landmarks. There are no `/content/ccta.nii.gz` placeholders and no file-upload step.

Research prototype only. Visually verify the RCA path before quantitative PCAT analysis.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Fresh copy of the exact experimental branch.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-prototype --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -e /content/OpenPlaque

from pathlib import Path
import os, numpy as np, SimpleITK as sitk, matplotlib.pyplot as plt
from scipy import ndimage as ndi
from openplaque.centerline import extract_rca_centerline, show_centerline_mip
from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series
print('OpenPlaque branch loaded: rca-centerline-prototype')


## Load the existing OpenPlaque study from Drive
This uses the same Drive layout as the UCLA plaque-characterization notebook: `MyDrive/OpenPlaque/Full_DICOM.zip` and outputs under `MyDrive/OpenPlaque/UCLA_Plaque_Characterization/`.

In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = ROOT / 'Full_DICOM.zip'
if not STUDY_ZIP.exists():
    raise FileNotFoundError(f'Missing {STUDY_ZIP}. Put Full_DICOM.zip in MyDrive/OpenPlaque.')

study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(
    study, fallback_series={'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
)
RCA_SERIES = int(series_map['RCA'])
ct_img, ct, _ = study.load_series(RCA_SERIES)
spacing_xyz = ct_img.GetSpacing()
print('Study:', STUDY_ZIP)
print('Detected series:', series_map)
print('RCA series:', RCA_SERIES)
print('CT shape:', ct.shape, 'spacing xyz mm:', spacing_xyz)


## Find the matching saved RCA mask automatically
The preferred file is `UCLA_Plaque_Characterization/masks/RCA_optimized_refined.nii.gz`. If that exact file is absent, the notebook searches `MyDrive/OpenPlaque` for RCA refined NIfTI masks and chooses the newest one whose array shape matches the RCA CT.

In [ ]:
preferred = ROOT / 'UCLA_Plaque_Characterization' / 'masks' / 'RCA_optimized_refined.nii.gz'
candidates = []
if preferred.exists(): candidates.append(preferred)
for p in ROOT.rglob('*.nii*'):
    n = p.name.lower()
    if 'rca' in n and 'refined' in n and 'class_map' not in n and p not in candidates:
        candidates.append(p)
candidates = sorted(candidates, key=lambda p: (p != preferred, -p.stat().st_mtime))

mask_path = None; mask_img = None; mask_raw = None
for p in candidates:
    try:
        im = sitk.ReadImage(str(p)); a = sitk.GetArrayFromImage(im)
        if a.shape == ct.shape and np.any(a > 0):
            mask_path, mask_img, mask_raw = p, im, a
            break
    except Exception as e:
        print('Skipping unreadable candidate:', p.name, repr(e))

if mask_path is None:
    found = '\n'.join(str(p) for p in candidates[:20]) or '(none)'
    raise FileNotFoundError('No matching RCA refined mask was found in MyDrive/OpenPlaque. Candidates searched:\n' + found)

print('Using RCA mask:', mask_path)
print('Mask labels:', np.unique(mask_raw))
print('Mask shape:', mask_raw.shape)


## Build an isolated RCA foreground and automatic endpoints
The refined mask contains the vessel/plaque labels for the RCA analysis. All nonzero voxels are treated as artery foreground; the largest connected component is retained. Endpoint selection is provisional and will be judged from the overlays.

In [ ]:
foreground = mask_raw > 0
labels, nlab = ndi.label(foreground, structure=ndi.generate_binary_structure(3,3))
sizes = ndi.sum(np.ones_like(foreground, dtype=np.uint8), labels, index=np.arange(1,nlab+1)) if nlab else np.array([])
if len(sizes) == 0: raise ValueError('RCA mask contains no connected foreground.')
largest_label = 1 + int(np.argmax(sizes))
rca = labels == largest_label
pts = np.argwhere(rca)
sp_zyx = np.asarray(spacing_xyz)[::-1]

# Farthest-pair approximation for the two ends of the RCA component.
p0 = pts[0]
d = np.linalg.norm((pts-p0)*sp_zyx, axis=1); p1 = pts[np.argmax(d)]
d = np.linalg.norm((pts-p1)*sp_zyx, axis=1); p2 = pts[np.argmax(d)]

# Provisional proximal choice: endpoint with larger array-x coordinate.
# This is deliberately visual-QC'd below; automatic ostium identification comes later.
if p2[2] > p1[2]: p1, p2 = p2, p1
OSTIUM_ZYX = tuple(int(v) for v in p1)
DISTAL_HINT_ZYX = tuple(int(v) for v in p2)
print('RCA foreground voxels:', int(rca.sum()))
print('Automatic ostium candidate zyx:', OSTIUM_ZYX)
print('Automatic distal candidate zyx:', DISTAL_HINT_ZYX)


In [ ]:
result = extract_rca_centerline(
    rca, spacing_xyz, OSTIUM_ZYX,
    distal_hint_zyx=DISTAL_HINT_ZYX,
    landmark_distances_mm=(0.0, 10.0, 50.0),
)
print(f'Centerline length: {result.length_mm:.1f} mm')
print('Snapped ostium:', result.ostium_zyx_voxel)
print('Endpoint:', result.endpoint_zyx_voxel)
print('Landmarks available:', sorted(result.landmarks_xyz_mm))
if 50.0 not in result.landmarks_xyz_mm:
    print('WARNING: extracted path is shorter than 50 mm.')


## Visual validation
A plausible result should continuously follow the main RCA, with no branch jump, and the 10 and 50 mm marks should both lie on the intended proximal-to-mid RCA.

In [ ]:
for axis, name in [(0,'axial MIP'), (1,'coronal MIP'), (2,'sagittal MIP')]:
    fig, ax = show_centerline_mip(ct, result, axis=axis)
    ax.set_title(f'{name}: provisional RCA centerline ({result.length_mm:.1f} mm)')
    plt.show()

ptsmm = result.points_xyz_mm
seg = np.linalg.norm(np.diff(ptsmm,axis=0),axis=1) if len(ptsmm)>1 else np.array([])
direct = np.linalg.norm(ptsmm[-1]-ptsmm[0]) if len(ptsmm)>1 else 0.0
print('Centerline voxels:', len(ptsmm))
print('Median/max step mm:', (float(np.median(seg)), float(np.max(seg))) if len(seg) else 'n/a')
print('Path/direct ratio:', round(result.length_mm/direct,3) if direct else 'n/a')
print('\nPASS only if: correct RCA, no branch jump, centerline near lumen center, and 10/50 mm landmarks are anatomically plausible.')
